<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day2/notebooks/1_embeddings_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Word Embeddings in Practice

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

In the lecture we saw the *idea* behind embeddings: represent each word as a dense vector,
learned from the company it keeps, so that similar words sit close together. Now we get our
hands on real embeddings.

By the end of this notebook you will be able to:

- Load **pre-trained** word embeddings (GloVe) and explore them
- Measure **similarity** and find **nearest neighbors**
- Do **word analogies** (king − man + woman ≈ ?)
- **Visualize** the embedding space in 2-D
- **Train your own** Word2Vec embeddings on a corpus, and see why domain matters

> Run each cell in order with `Shift + Enter`. Try each **✏️ Exercise** before moving on.
> A couple of cells download data from the internet, so give them a moment.


## 0. Setup

We use **gensim**, the standard Python library for word embeddings. It is usually
pre-installed in Colab; the line below makes sure.


In [ ]:
!pip install gensim --q

import numpy as np
import matplotlib.pyplot as plt

import gensim.downloader as api
from gensim.models import Word2Vec

print("Setup complete!")

## 1. Loading pre-trained embeddings

Rather than train from scratch (which needs a huge corpus), we start with embeddings that
someone else already trained on billions of words. We use **GloVe**, trained on Wikipedia
and news text.

We pick a small version (50 dimensions, ~66 MB) so it downloads quickly. The first run may
take a minute.


In [ ]:
# Download pre-trained GloVe vectors (50-dimensional).
# This downloads ~66 MB the first time.
glove = api.load("glove-wiki-gigaword-50")

print("Loaded!")
print("Vocabulary size:", len(glove))
print("Vector size (dimensions):", glove.vector_size)

Let's look at an actual word vector. Each word is now just a list of numbers.


In [ ]:
vec = glove["king"]
print("The vector for 'king' has", len(vec), "dimensions.")
print()
print("First 10 numbers:")
print(vec[:10])

That is the whole idea made concrete: the word ``king`` *is* these 50 numbers. On their
own they are meaningless — what matters is how they relate to other words' numbers.


> **✏️ Exercise 1**
>
> Pull out the vector for a word of your choice. What are its dimensions? Confirm it has the
> same length as the vector for ``king``. *(Try a word you'd expect to be in Wikipedia.)*


In [ ]:
# Your code here


## 2. Similarity and nearest neighbors

The payoff of embeddings: similar words have similar vectors. gensim measures this with
**cosine similarity** and can find the **nearest neighbors** of any word.


In [ ]:
# How similar are two words? (1.0 = identical direction, 0 = unrelated)
print("king  vs queen: ", round(glove.similarity("king", "queen"), 3))
print("king  vs man:   ", round(glove.similarity("king", "man"), 3))
print("king  vs banana:", round(glove.similarity("king", "banana"), 3))

Exactly what we would hope: ``king`` is close to ``queen`` and ``man``, but far from
``banana``. The model learned this purely from context — nobody labeled these relationships.

Now the most useful query: **what are the nearest neighbors of a word?**


In [ ]:
# The 10 words most similar to "university"
for word, score in glove.most_similar("university", topn=10):
    print(f"  {word:<15} {score:.3f}")

> **✏️ Exercise 2**
>
> Find the 10 nearest neighbors of a word relevant to *your* research (e.g. ``policy``,
> ``media``, ``health``, ``gender``). Do the neighbors make sense? Do any surprise you?


In [ ]:
# Your code here


## 3. Word analogies

The famous result: because meaning is encoded geometrically, we can do *arithmetic* with
words. ``king − man + woman`` lands near ``queen``.

In gensim, `positive` words are added and `negative` words subtracted.


In [ ]:
# king - man + woman = ?
result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("king - man + woman ≈")
for word, score in result:
    print(f"  {word:<12} {score:.3f}")

It works! The top answer is ``queen``. Let's try another kind of relationship —
country to capital city.


In [ ]:
# paris - france + germany = ? (capital-of relationship)
result = glove.most_similar(positive=["paris", "germany"], negative=["france"], topn=3)
print("paris - france + germany ≈")
for word, score in result:
    print(f"  {word:<12} {score:.3f}")

> **✏️ Exercise 3**
>
> Try your own analogy. Some to get you started:
> - ``good − better + bad`` (comparative)
> - ``walking − walk + swim`` (verb tense)
> - Or invent one from your own domain.
>
> Not all analogies work cleanly — note one that *fails* and think about why.


In [ ]:
# Your code here


## 4. Visualizing the embedding space

Embeddings live in many dimensions (50 here), which we cannot see. We use **PCA** to project
a handful of words down to 2-D and plot them. Watch how related words cluster.


In [ ]:
from sklearn.decomposition import PCA

# Pick some words from a few clear categories
words = [
    "king", "queen", "prince", "princess",       # royalty
    "dog", "cat", "horse", "cow",                 # animals
    "paris", "london", "berlin", "tokyo",         # cities
    "happy", "joyful", "sad", "angry",            # emotions
]

# Get their vectors and project to 2-D with PCA
vectors = np.array([glove[w] for w in words])
coords = PCA(n_components=2).fit_transform(vectors)

# Plot
plt.figure(figsize=(11, 8))
plt.scatter(coords[:, 0], coords[:, 1], color="#34B233", s=60)
for i, word in enumerate(words):
    plt.annotate(word, (coords[i, 0], coords[i, 1]),
                 fontsize=12, xytext=(5, 5), textcoords="offset points")
plt.title("Word embeddings projected to 2-D (PCA)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

You should see the categories forming their own clusters — royalty in one area, cities
in another, and so on. Remember from the lecture: 2-D is a **lossy cartoon** of the real
50-D space, so treat it as intuition, not proof.


> **✏️ Exercise 4**
>
> Make your own version of the plot with words from **two or three categories you care
> about** (e.g. political terms, health terms, emotions). Do they separate into clusters?


In [ ]:
# Your code here


## 5. Training your own embeddings

Pre-trained vectors are trained on general text (Wikipedia, news). But sometimes your
domain uses words differently, or uses words that don't appear in general corpora. In that
case you can **train your own** embeddings on your own corpus.

To keep it fast and self-contained, we train on a small built-in corpus. In real research
you would use a much larger one (thousands to millions of sentences).


In [ ]:
import nltk
nltk.download("brown")
from nltk.corpus import brown

# The Brown corpus: a classic collection of English sentences.
# gensim's Word2Vec wants a list of tokenized sentences (list of lists of words).
sentences = [[w.lower() for w in sent] for sent in brown.sents()]

print("Number of sentences:", len(sentences))
print("Example sentence:", sentences[0][:12], "...")

Now we train a **Word2Vec** model. The key parameters:

- `vector_size`: how many dimensions each word vector has
- `window`: how many neighboring words count as "context"
- `min_count`: ignore words appearing fewer than this many times
- `sg`: 1 for skip-gram, 0 for CBOW (from the lecture!)


In [ ]:
# Train a skip-gram Word2Vec model on the Brown corpus
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,          # skip-gram
    epochs=5,
    seed=42,
)

print("Training complete!")
print("Vocabulary size:", len(model.wv))

Our home-grown embeddings work just like the pre-trained ones — we can query similarity
and neighbors. They will be rougher, because Brown is tiny compared to Wikipedia.


In [ ]:
# Nearest neighbors in OUR trained model
word = "money"
if word in model.wv:
    print(f"Words most similar to '{word}' in our Brown-trained model:")
    for w, score in model.wv.most_similar(word, topn=8):
        print(f"  {w:<15} {score:.3f}")
else:
    print(f"'{word}' not in vocabulary — try another.")

> **✏️ Exercise 5**
>
> Compare the **same word** in the two models: look up its nearest neighbors in
> `glove` (pre-trained) and in `model.wv` (your Brown-trained one). How do they differ?
> Which looks higher quality, and why might that be?
>
> *(Pick a reasonably common word so it's in both vocabularies.)*


In [ ]:
# Your code here


> **✏️ Exercise 6**
>
> Re-train the model with **CBOW** instead of skip-gram (`sg=0`) and/or a larger `window`.
> Do the nearest neighbors change? There is no single 'correct' setting — it depends on your
> corpus and goal.


In [ ]:
# Your code here


## 6. A short detour: measuring bias

Before the wrap-up, a quick demonstration of something we return to in the next part of the
lecture. Embeddings learn from human text, so they absorb the **social biases** in that text.
That is a serious hazard — but it also means we can *measure* those biases.

Here we build a tiny **gender-association score**: for a given word, is it closer to
"male" words or "female" words in the embedding space? We reuse the pre-trained `glove`
vectors from earlier.


In [ ]:
# Two small sets of clearly gendered anchor words
male_words = ["he", "him", "his", "man", "male"]
female_words = ["she", "her", "hers", "woman", "female"]

def gender_association(word, model=glove):
    """Positive = leans male, negative = leans female (by cosine similarity)."""
    male_sim = np.mean([model.similarity(word, m) for m in male_words])
    female_sim = np.mean([model.similarity(word, f) for f in female_words])
    return male_sim - female_sim

# Sanity check on the anchors themselves
print("Association score (positive = male-leaning, negative = female-leaning):")
print("  king: ", round(gender_association("king"), 3))
print("  queen:", round(gender_association("queen"), 3))

``king`` should score positive (male-leaning) and ``queen`` negative (female-leaning) —
a quick check that our measure behaves sensibly. Now let's point it at something more telling:
**professions**, which carry no *inherent* gender.


In [ ]:
professions = [
    "nurse", "teacher", "librarian", "receptionist",
    "engineer", "scientist", "programmer", "surgeon",
    "doctor", "lawyer", "professor", "assistant",
]

scores = [(p, gender_association(p)) for p in professions]
scores.sort(key=lambda x: x[1])   # most female-leaning first

print(f"{'profession':<14}{'score':>8}   leaning")
print("-" * 40)
for prof, score in scores:
    lean = "male" if score > 0 else "female"
    print(f"{prof:<14}{score:>8.3f}   {lean}")

Look at the pattern. Professions like ``nurse`` and ``receptionist`` lean female, while
``engineer`` and ``programmer`` lean male — even though nothing about these jobs is
intrinsically gendered. The embedding has absorbed **occupational gender stereotypes** from
its training text.

Let's visualize it as a simple bar chart.


In [ ]:
profs = [p for p, s in scores]
vals = [s for p, s in scores]
colors = ["#34B233" if v < 0 else "#1A1A2E" for v in vals]

plt.figure(figsize=(10, 6))
plt.barh(profs, vals, color=colors)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Gender association of professions in GloVe\n(left = female-leaning, right = male-leaning)")
plt.xlabel("male-leaning  ->")
plt.tight_layout()
plt.show()

> **✏️ Exercise 7**
>
> This is a crude measure. Try it with your own set of words — for example adjectives
> (``strong``, ``gentle``, ``ambitious``, ``caring``) or fields of study. What patterns show
> up? Then think critically: what are the **limitations** of measuring bias this way, and
> what would you need to do before making a research claim from it?
>
> *(We discuss validation and pitfalls right after this notebook.)*


In [ ]:
# Your code here


## Wrap-up

You have now worked with real embeddings, both ways:

- Loaded **pre-trained GloVe** vectors and explored similarity, neighbors, and analogies
- **Visualized** the space in 2-D and saw meaningful clusters
- **Trained your own** Word2Vec model and compared it to the pre-trained one

Key takeaways:

- Embeddings turn words into dense vectors where **distance reflects meaning**
- **Pre-trained** vectors are a great default for general language
- **Training your own** makes sense when your domain language is special — but needs a large
  corpus to be any good (our Brown example is deliberately small, hence rough)

### Optional challenge

Compute a document vector by **averaging** the word vectors in a sentence (from the lecture:
the simplest way to go from word embeddings to a document embedding). Write a short function
`sentence_vector(sentence)` that averages the GloVe vectors of the words in a sentence, then
use it to measure the similarity between two sentences of your choice.

*(Hint: `np.mean([...], axis=0)`, and skip words not in the vocabulary.)*


In [ ]:
# Optional challenge — your code here
